# 02 — MLP (Multi-Layer Perceptron)
Run `01_eda_preprocessing.ipynb` first to generate `data/cleaned.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
df = pd.read_csv('data/cleaned.csv')
feature_cols = [c for c in df.columns if c not in ['Customer_ID','Credit_Score_Enc']]
agg_df = df.groupby('Customer_ID')[feature_cols].mean()
target_df = df.groupby('Customer_ID')['Credit_Score_Enc'].agg(lambda x: x.mode()[0])
agg_df['target'] = target_df

X = agg_df.drop(columns=['target']).values
y = agg_df['target'].values
X = StandardScaler().fit_transform(X)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
], name='MLP')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=20, batch_size=64, verbose=1)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=['Poor','Standard','Good']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train'); axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('MLP Training vs Validation Accuracy'); axes[0].legend()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Poor','Standard','Good'], yticklabels=['Poor','Standard','Good'])
axes[1].set_title('MLP Confusion Matrix')
plt.tight_layout(); plt.savefig('results/mlp_results.png', dpi=150); plt.show()